# Retrieval-Augmented Generation (RAG)

In [1]:
import chromadb
import dotenv
from pathlib import Path
from agents import Agent, Runner, function_tool, trace

dotenv.load_dotenv()

True

Create a static calorie table that we can use as a tool:

In [ ]:
# We populated the RAG with the data from the data/calories.csv file in
# the rag_setup.ipynb notebook
chroma_client = chromadb.PersistentClient(path= "../chroma")
nutrition_db = chroma_client.get_collection(name="nutrition_db")
nutrition_qna = chroma_client.get_collection(name="nutrition_qna")

In [18]:
@function_tool
def calorie_lookup_tool(query: str, max_results: int = 3) -> str:
    """
    Tool function for a RAG database to look up calorie information for specific food items, but not for meals.

    Args:
        query: The food item to look up.
        max_results: The maximum number of results to return.

    Returns:
        A string containing the nutrition information.
    """

    results = nutrition_db.query(query_texts=[query], n_results=max_results)

    if not results["documents"][0]:
        return f"No nutrition information found for: {query}"

    # Format results for the agent
    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        food_item = metadata["food_item"].title()
        calories = metadata["calories_per_100g"]
        category = metadata["food_category"].title()

        formatted_results.append(
            f"{food_item} ({category}): {calories} calories per 100g"
        )

    return "Nutrition Information:\n" + "\n".join(formatted_results)


@function_tool
def nutrition_lookup_tool(query: str, max_results: int = 2 ) -> str:
    """
    Nutrition look up tool for malnutrition in human and pregnant woman 

    Args:
        query:  The symptoms for malnutrition.
        max_results: The maximum number of results to return.

    Returns:
        A string containing the malnutrition information        
       
    """
    results = nutrition_qna.query(query_texts= [query] , n_results=max_results )

    if not results['documents'][0]:
        return f"No information available for the query : {query}"
    
    formatted_results = []
    for i, doc in enumerate(results['documents'][0]):
        formatted_results.append(doc)

    return f"The Information are for your query are : \n" + "\n".join(formatted_results)    

Let's test this out: 

_The following cell only works before you add the `@function_tool` annotation to `calorie_lookup_tool` function_

In [ ]:
# calorie_lookup_tool('bananas')

In [19]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information and nutrition advice.
    You give concise answers.
    If you need to look up calorie information, use the calorie_lookup_tool.
    If you need to look up nutrition,always use the nutrition_lookup_tool first, to check if there are any information in the knowledge base.
    """,
    tools =[calorie_lookup_tool, nutrition_lookup_tool]
)

In [20]:
with trace("Nutrition Assistant for Nutri and Cal details with RAG"):
    result = await Runner.run(
        calorie_agent,
        "What are the best meal choices for pregnant women and how many calories do they have?",
        #[{"role": "user", "content": "How many calories are in a banana and an apple? max_results=2"}],
    )
    print(result.final_output)

Here are concise guidelines and calorie context for pregnant women:

- Key foods to prioritise
  - Iron-rich: lean meat, poultry, fish, eggs, beans, lentils, fortified cereals
  - Folate: leafy greens, citrus fruits, beans, fortified grains
  - Calcium: dairy or fortified non-dairy alternatives
  - Protein: lean meats, fish, eggs, dairy, legumes, nuts, seeds
  - DHA/EPA: fatty fish (low-mercury, e.g., salmon), fortified foods, or supplements if advised
  - Fiber and fluids: whole grains, fruits, vegetables, water
- Calorie note
  - Typically an extra ~200 kcal per day during pregnancy, plus a balanced overall diet
- Practical meal ideas (approximate additions)
  - Breakfast: yogurt parfait with fruit and whole-grain granola (~350–450 kcal)
  - Lunch: iron-rich turkey sandwich with greens and a side of fruit (~400–500 kcal)
  - Snack: glass of milk or fortified yogurt with a piece of fruit (~150–200 kcal)
  - Dinner: grilled fish, quinoa or beans, and veggies (~450–600 kcal)
- Important